# 03 — Train Model on Colab GPU

This notebook runs the full training pipeline on a free T4 GPU.

**Before starting:**
1. Download `surface_crack_data.zip` (already generated — 228 MB)
2. Run the cells below — the upload widget will prompt you to select the zip

**Runtime:** Runtime → Change runtime type → Hardware accelerator = **T4 GPU**

In [ ]:
# Step 0: Upload dataset zip
from google.colab import files
print("⬆️ Click 'Choose File' and select surface_crack_data.zip")
uploaded = files.upload()
DATA_ZIP_PATH = "/content/" + list(uploaded.keys())[0]
print(f"✅ Uploaded: {DATA_ZIP_PATH}")

In [ ]:
import os, shutil, zipfile

REPO_URL = "https://github.com/esetu-git-public/bootcamp-ace-26-team-7.git"

# Clone repo
if not os.path.exists("/content/bootcamp"):
    !git clone {REPO_URL} /content/bootcamp
%cd /content/bootcamp

In [ ]:
# Extract dataset — class folders go directly into data/raw/
os.makedirs("data/raw", exist_ok=True)
with zipfile.ZipFile(DATA_ZIP_PATH, "r") as z:
    z.extractall("data/raw")
print("data/raw:", os.listdir("data/raw"))

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# Verify GPU is active
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Check class distribution
from src.config import Config

for class_name in Config.CLASSES:
    class_dir = os.path.join(Config.RAW_DATA_DIR, class_name)
    if os.path.exists(class_dir):
        images = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"{class_name}: {len(images)} images")
    else:
        print(f"{class_name}: NOT FOUND")

In [ ]:
# Step 1: Prepare data — stratified split
!python -m src.prepare_data
print("\nTrain set:", os.listdir("data/processed/train"))

In [ ]:
# Step 2: Train model (warmup 5 + finetune 15 epochs on T4 GPU ≈ 2-3 min)
!python -m src.train

In [ ]:
# Step 3: Evaluate on test set
!python -m src.evaluate

In [ ]:
# Step 4: Download model and reports
from google.colab import files

model_path = "models/best_model.pth"
if os.path.exists(model_path):
    files.download(model_path)

if os.path.exists("reports"):
    for f in os.listdir("reports"):
        files.download(os.path.join("reports", f))